# E3の結果を小分けにする（直し）
本番が全部終わってから、CPUのままで「すべてのセルを実行」。1分ほど。

In [ ]:
# E3_main.json を 3MB 以下の小分けにする（こちらで読むため。中身は変えない）
from google.colab import drive
drive.mount("/content/drive")
import json, os
D = "/content/drive/MyDrive/AKANE_アーキテクチャ/E3_results"
res = json.load(open(f"{D}/E3_main.json"))
items = res["items"]
print(len(items), "件 / NLI済み", sum("nli" in it for it in items))
skip = {"evidence", "sents", "paras"}   # 資料の文はこちらの手元データ(id)から戻せるので外す
small = [{k: v for k, v in it.items() if k not in skip} for it in items]
parts, cur, size = [], [], 0
for it in small:
    n = len(json.dumps(it, ensure_ascii=False).encode()) + 2
    if cur and size + n > 3_000_000:
        parts.append(cur); cur, size = [], 0
    cur.append(it); size += n
parts.append(cur)
for i, p in enumerate(parts):
    f = f"{D}/E3_main_part{i+1}.json"
    json.dump({"meta": res["meta"], "part": i+1, "n_parts": len(parts), "items": p}, open(f, "w"), ensure_ascii=False)
    print(f, len(p), "件", round(os.path.getsize(f)/1e6, 1), "MB")
print("できた")
